# OpenNeuro ds003517 / 8-bit Gameplay

Dataset key: `ds003517_sub001_derived`

Source: OpenNeuro `ds003517`, subject `sub-001`. The import/build command is source-specific and visible in this notebook; plotting uses existing Week 15 8-bit helpers.

In [ ]:
import Pkg

function find_repo_root()
    candidates = unique(normpath.([
        pwd(),
        joinpath(pwd(), ".."),
        joinpath(pwd(), "..", ".."),
        joinpath(pwd(), "..", "..", ".."),
    ]))
    for candidate in candidates
        if isdir(joinpath(candidate, "notebooks")) && isdir(joinpath(candidate, "scripts"))
            return candidate
        end
    end
    error("Could not locate repository root from pwd=$(pwd()).")
end

const REPO_ROOT = find_repo_root()
const NOTEBOOK_DIR = joinpath(REPO_ROOT, "notebooks", "week_19", "data_sources")
const DATASETS_ROOT = joinpath(REPO_ROOT, "notebooks", "datasets")
const WEEK19_DOWNLOADS = joinpath(REPO_ROOT, "notebooks", "week_19", "downloads")
const PYTHON = begin
    venv_python = joinpath(REPO_ROOT, ".venv_8bit", "bin", "python")
    isfile(venv_python) ? venv_python : "python"
end

Pkg.activate(joinpath(REPO_ROOT, "notebooks", "model_test"))

using CairoMakie
using CSV
using DataFrames
using HDF5
using JSON3
using Printf
using Statistics

include(joinpath(REPO_ROOT, "notebooks", "week_15", "try_new_data_helpers.jl"))
using .Week15TryNewData

mkpath(WEEK19_DOWNLOADS)
RNG_SEED = Int(mod(time_ns(), UInt64(typemax(Int))))
println("Repo root: ", REPO_ROOT)
println("Python: ", PYTHON)
println("RNG seed: ", RNG_SEED)

include(joinpath(REPO_ROOT, "notebooks", "week_15", "sort_variable_overview_helpers.jl"))
using .Week15SortVariableOverview

In [ ]:
const DATASET_KEY = "ds003517_sub001_derived"
const SOURCE_DATASET = "ds003517"
const RUN_IMPORT = false
const PREPARE_SCRIPT = joinpath(REPO_ROOT, "scripts", "prepare_openneuro_8bit_dataset.py")
const SOURCE_ROOT = joinpath(DATASETS_ROOT, SOURCE_DATASET)
const OUTPUT_DIR = joinpath(DATASETS_ROOT, DATASET_KEY)
const REQUIRED_FILES = [joinpath(OUTPUT_DIR, "epochs.hdf5"), joinpath(OUTPUT_DIR, "events.csv"), joinpath(OUTPUT_DIR, "metadata.json")]

@assert isfile(PREPARE_SCRIPT) "Missing importer: $PREPARE_SCRIPT"

if all(isfile, REQUIRED_FILES)
    println("Bundle already ready: ", OUTPUT_DIR)
elseif RUN_IMPORT
    run(Cmd([PYTHON, PREPARE_SCRIPT, "--source-root", SOURCE_ROOT, "--output-dir", OUTPUT_DIR]))
else
    @info "RUN_IMPORT=false; not running importer. Set RUN_IMPORT=true to rebuild/download."
end

In [ ]:
data_bundle = load_8bit_overview_data()
println("8bit EEG shape: ", data_bundle.full_shape)
println("Events rows: ", nrow(data_bundle.events))

sort_summary = summarize_sort_columns(data_bundle)
display(sort_summary)
sort_columns = plottable_sort_columns(data_bundle)

sort_audit = overview_sort_order_audit_df(data_bundle; sort_columns = sort_columns)
display(sort_audit)
@assert all(sort_audit.status .== "ok") "Sort-order audit failed for ds003517."


In [ ]:
N_CHANNELS_PER_SORT = 2
plot_all_sort_variable_figures(data_bundle;
    sort_columns = sort_columns,
    n_channels = N_CHANNELS_PER_SORT,
    seed = RNG_SEED,
)
